# L52 · Audio Core 完结 — 特征工程收口，38 个单元测试全绿，面试证据整理

**目标**：跑通 `tests/audio/` 全部测试，分析性能，完成 Month 1 通关标志。

🔗 Aurora 连接：
- `src/aurora/audio/io.py` — WAV 读写
- `src/aurora/audio/windows.py` — Hann / Hamming
- `src/aurora/audio/transforms.py` — DFT / FFT / IFFT / RFFT
- `src/aurora/audio/stft.py` — 分帧（framing） + 短时傅里叶变换
- `src/aurora/audio/mel.py` — Hz↔Mel 转换、mel 滤波器组、mel 频谱
- `src/aurora/audio/mfcc.py` — DCT-II、MFCC

这节课的核心直觉：**从一段 WAV 到 13 个 MFCC 系数，每一步都有数学公式支撑，而测试套件就是把公式变成断言**。
`atol=1e-8` 级别的数值比对意味着任何一行实现偏差——错误的归一化系数、off-by-one 的帧数——都会让测试立刻报红。
Month 1 的通关标准不是"记住代码"，而是"能对着空白文件重写任意函数并解释每行的数学含义"。

In [ ]:
import subprocess
import numpy as np

## 1. 从信号到特征的完整管线

```
WAV → 分帧(frame_signal) → 加窗(hann/hamming) → FFT(rfft)
    → |·|² → mel 滤波(mel_filterbank) → log → DCT-II → MFCC
```

每一步的数学形式：

- **分帧**：把长为 `T` 的信号切成长 `N`、步长 `H` 的帧，共 `1 + (T-N)//H` 帧。
- **加窗**：逐元素乘以 `w[n] = 0.5 * (1 - cos(2π n / N))`（Hann 周期窗）。
- **RFFT**：`X[k] = sum_{n=0}^{N-1} x[n] * e^{-j 2π k n / N}`，保留 `N//2+1` 个非冗余频点。
- **Mel 滤波**：`M[m] = sum_k fb[m,k] * |X[k]|²`，三角滤波器组（triangular filter bank），`fb` 形状 `(n_mels, N//2+1)`。
- **对数压缩（log compression）**：`log_mel = 10 * log10(M + eps)`，模拟人耳响度感知的对数特性。
- **DCT-II（正交归一化）**：`C[i] = scale[i] * sum_m log_mel[m] * cos(π/M * (m+0.5) * i)`，前 13 个系数即 MFCC。

In [ ]:
from aurora.audio import sine, stft, mel_spectrogram, mfcc

sr = 16000
x = sine(440.0, 1.0, sr)          # 1 秒 440 Hz 纯音

S = stft(x, n_fft=1024, hop_length=256)          # (n_frames, 513) complex
M = mel_spectrogram(x, sr, n_fft=1024, hop_length=256, n_mels=80)  # (n_frames, 80)
C = mfcc(x, sr, n_mfcc=13, n_fft=1024, hop_length=256)             # (n_frames, 13)

print(f"STFT  shape: {S.shape}  dtype: {S.dtype}")
print(f"Mel   shape: {M.shape}  dtype: {M.dtype}")
print(f"MFCC  shape: {C.shape}  dtype: {C.dtype}")
print(f"MFCC[0, :5] = {C[0, :5].round(4)}")
assert np.iscomplexobj(S)
assert M.shape[1] == 80
assert C.shape[1] == 13
print("✅ 管线形状全部正确")

## 2. 测试套件验证

`tests/audio/` 下有两个测试文件：

| 文件 | 覆盖范围 | 关键断言 |
|------|----------|----------|
| `test_transforms.py` | DFT / FFT / IFFT / RFFT | `atol=1e-9` 对比 `numpy.fft` |
| `test_features.py` | windows / STFT / mel / MFCC / WAV I/O | `atol=1e-12`窗函数（window function）、`atol=1e-8`（MFCC）|

关键测试点举例：

```python
# FFT 和 NumPy 逐元素一致
np.testing.assert_allclose(T.fft(x), np.fft.fft(x), atol=1e-9)

# DCT-II 手动公式
basis = np.cos(np.pi / n * np.outer(k, k + 0.5))
ref = x @ basis.T * scale
np.testing.assert_allclose(dct_ii(x), ref, atol=1e-12)

# Hann 窗与 numpy.hanning 一致（对称模式）
np.testing.assert_allclose(hann(64, periodic=False), np.hanning(64), atol=1e-12)
```

这些测试的设计原则：每个函数都有一个独立于实现的参考答案（公式或 NumPy），只要数学正确就一定通过。

In [ ]:
import sys
from pathlib import Path

result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/audio/", "-v", "--tb=short", "--no-header"],
    capture_output=True, text=True,
    cwd=str(Path.cwd().parent.parent),  # aurora repo root
)
print(result.stdout[-4000:] if len(result.stdout) > 4000 else result.stdout)
if result.returncode != 0:
    print(result.stderr[-2000:])
assert result.returncode == 0, "❌ 有测试失败，请检查上方输出"
print("✅ tests/audio/ 全部通过")

## 3. Month 1 交付标准

通关的三层验证：

**层1 — 代码能跑**：`make test` 零失败，`make demo` 输出正确频谱图。

**层2 — 数学能说清**：能对着空白文件重写任意一个函数。逐行对应公式，不依赖记忆：
- `dct_ii`：`basis[k, m] = cos(π/N * (m+0.5) * k)`，正交归一化系数 `sqrt(2/N)`，首项 `sqrt(1/N)`。
- `mel_filterbank`：三角滤波器顶点坐标在 mel 域均匀分布，转换回 Hz 后不均匀。
- `fft`：Cooley-Tukey 蝶形分解，`N=2^k` 时将 `O(N²)` 降为 `O(N log N)`。

**层3 — 能对比差异**：能解释 aurora 和 librosa 输出微小不同的原因（归一化策略、mel 刻度参数、中心填充行为）。

```
检查清单
[ ] pytest tests/audio/  →  全绿
[ ] 能手写 dct_ii(x)，解释每行
[ ] 能手写 mel_filterbank(n_mels, n_fft, sr)，解释三角形顶点坐标
[ ] 能解释 hann(N, periodic=True) 和 periodic=False 的区别
[ ] 能说出 stft 输出形状 (n_frames, N//2+1) 的来由
```

In [ ]:
from aurora.audio import (
    hann, hamming, dct_ii, mel_filterbank,
    frame_signal, stft, mel_spectrogram, mfcc,
    hz_to_mel, mel_to_hz, read_wav, write_wav
)

# 检查1：DCT-II 手动验证
x_test = np.array([1.0, 2.0, 3.0, 4.0])
n = len(x_test)
k = np.arange(n)
basis = np.cos(np.pi / n * np.outer(k, k + 0.5))
scale = np.full(n, np.sqrt(2.0 / n)); scale[0] = np.sqrt(1.0 / n)
ref_dct = x_test @ basis.T * scale
np.testing.assert_allclose(dct_ii(x_test), ref_dct, atol=1e-12)
print("✅ DCT-II 手动公式验证通过")

# 检查2：mel 滤波器组形状和非负性
fb = mel_filterbank(n_mels=40, n_fft=1024, sample_rate=16000)
assert fb.shape == (40, 513) and np.all(fb >= 0.0)
print(f"✅ mel_filterbank shape={fb.shape}, 全非负, 各行峰值 > 0: {np.all(fb.max(axis=1) > 0)}")

# 检查3：Hann 窗周期/对称
per = hann(8, periodic=True)
sym = hann(9, periodic=False)
np.testing.assert_allclose(per, sym[:-1], atol=1e-12)
print("✅ hann periodic=True 等于 symmetric N+1 去掉最后一点")

# 检查4：STFT 形状
x_1s = sine(440.0, 1.0, 16000)
S = stft(x_1s, n_fft=1024, hop_length=256)
assert S.shape[1] == 1024 // 2 + 1
print(f"✅ stft 输出 {S.shape}，频率维 = N//2+1 = {1024//2+1}")

# 检查5：mel 转换可逆
freqs = np.array([0.0, 100.0, 440.0, 1000.0, 8000.0])
np.testing.assert_allclose(mel_to_hz(hz_to_mel(freqs)), freqs, atol=1e-6)
print("✅ hz_to_mel → mel_to_hz 往返误差 < 1e-6 Hz")

## 参数实验：aurora.mfcc vs librosa.feature.mfcc

**参数**：`n_fft=1024, hop_length=256, n_mels=80, n_mfcc=13`，信号为 1 秒 440 Hz 纯音。

**预期现象**：
- 形状相同 `(n_frames, 13)`，数值相近但不完全一致。
- 主要差异来源：
  1. `center` 参数：librosa 默认 `center=True`（首尾各填充 `n_fft//2` 个零），aurora 默认 `center=False`，导致帧数不同。
  2. mel 刻度：librosa 默认 HTK mel 公式 `2595 * log10(1 + f/700)`，aurora 一致，但归一化模式（`norm="slaney"` vs 无）可能不同。
  3. power 参数：librosa 默认对功率谱（`|X|²`）建 mel，aurora `mel_spectrogram` 同样是功率谱，应一致。

运行下方 code 格，观察 RMSE 数量级和第 0 帧差异。

In [ ]:
try:
    import librosa
    HAS_LIBROSA = True
except ImportError:
    HAS_LIBROSA = False
    print("librosa 未安装，跳过对比（pip install librosa 后重试）")

if HAS_LIBROSA:
    from aurora.audio import sine, mfcc as aurora_mfcc

    sr = 16000
    x = sine(440.0, 1.0, sr).astype(np.float32)

    # aurora (center=False，和 librosa center=True 帧数会不同)
    C_aurora = aurora_mfcc(x.astype(np.float64), sr,
                           n_mfcc=13, n_fft=1024, hop_length=256, n_mels=80)

    # librosa (center=True by default)
    C_lib_center = librosa.feature.mfcc(
        y=x, sr=sr, n_mfcc=13, n_fft=1024, hop_length=256, n_mels=80
    ).T  # librosa 返回 (n_mfcc, n_frames), 转置为 (n_frames, n_mfcc)

    print(f"aurora  MFCC shape: {C_aurora.shape}")
    print(f"librosa MFCC shape (center=True): {C_lib_center.shape}")

    # 对齐帧数后比较
    n_common = min(C_aurora.shape[0], C_lib_center.shape[0])
    rmse = np.sqrt(np.mean((C_aurora[:n_common] - C_lib_center[:n_common])**2))
    print(f"前 {n_common} 帧 RMSE = {rmse:.4f}")
    print(f"aurora  第0帧前5系数: {C_aurora[0, :5].round(3)}")
    print(f"librosa 第0帧前5系数: {C_lib_center[0, :5].round(3)}")
    print()
    print("差异来源分析：")
    print(f"  帧数差 = {C_lib_center.shape[0] - C_aurora.shape[0]}（来自 center=True 填充）")
    print("  系数差异主要在 C[0]（能量项）：librosa 默认 norm=None，aurora 不填充")

## 本课收束

`mfcc()`、`mel_filterbank()`、`dct_ii()`、`stft()` 构成 Aurora Audio Analysis Engine 的完整特征提取栈，所有函数已通过 `tests/audio/` 数值断言（`atol` 最严达 `1e-12`）验证与 NumPy / 公式参考一致。
Month 1 的交付物是一个**零黑盒的纯 NumPy 音频特征管线**，覆盖从 WAV I/O 到 13 维 MFCC 向量的全部步骤。
`aurora.audio` 模块已可作为 L53+ 深度学习模块的特征提取后端，直接调用 `mfcc()` 输出 `(n_frames, 13)` 张量。
下一课（L53）将用图形化方式展示 MFCC 的完整提取流水线：波形 → 声谱图 → Mel 谱 → 倒谱系数，逐层可视化。